# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.

   Answer: No. A vision-language-action model successfully completing a coffee delivery task 100 times does not prove that it can perform the task in every home environment. Repeated successful demonstrations show that the model can perform the task under the conditions represented in the training data. However, they do not guarantee that the model understands the underlying physical constraints or can generalise to unfamiliar situations.

   For example, changes in the environment, mug shape, liquid fill level, obstacles or human position may affect the outcome. A model may reproduce previously learned actions without being able to predict whether those actions will remain safe under changed conditions.
   
   Therefore, successful demonstrations provide evidence of performance within specific conditions, but they do not establish general competence. Reliable competence requires testing across different situations, predicting possible outcomes and verifying that the robot can adapt when conditions change.
   
   
   Confidence: 5/5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.

   Answer: If the white mug changes from half-full to nearly full, the coffee may spill when the robot turns because the liquid moves due to acceleration and changes in direction.A nearly full mug has less empty space available to accommodate liquid movement. Consequently, the liquid may reach the rim even when the robot follows the same trajectory that was successful with a half-full mug.
   
   The untested assumption in the original demonstration is that the same movement parameters remain safe regardless of the liquid fill level.The robot must consider the mug's fill level, turning velocity and trajectory before performing the movement. It should predict the resulting liquid wave height and adjust its motion if necessary.
   
   Therefore, the successful half-full mug demonstration does not establish that the same carrying method will work safely with a nearly full mug.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.

   - Answer: The robot should ask Eva for confirmation before proceeding.
   
   The blue mug is unavailable, and although the white mug is physically available, Eva's acceptance of the alternative mug has not been confirmed. The missing evidence is explicit confirmation from Eva that she accepts the white mug as a replacement for her usual blue mug. The robot must distinguish between physical availability and user acceptance. The availability of an object does not automatically mean that it is an acceptable choice for the user.
   Therefore, the robot should ask:
   "Your blue mug is in the dishwasher. Is the white one all right?"
   The robot should proceed only after receiving confirmation and reassessing the remaining requirements of the coffee delivery task.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.

   - Answer: 

**T1 – Explain:** Why does the mug not spill? Is it because the robot turns slowly or because the mug is only half-full?

**T2 – Predict:** Will a nearly full mug spill when the robot follows the same route?

**T3 – Design:** Would using a different gripper or a tray change the way the robot carries the mug?

**T4 – Bound:** For which types of mugs, liquid levels, and routes does the robot's method work?

**T5 – Check:** Does the recorded information include Eva's answer, the robot's movement, and the final outcome?
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
   - Answer:
  

| Component | Responsibility |
|---|---|
| Interpret and Ground | Understands the user's request and identifies the objects and goals. |
| Know and Predict | Uses the Semantic Digital Twin to understand the environment and predict outcomes. |
| Perceive | Collects sensor information and updates the robot's belief about the world. |
| Propose | Generates possible actions using learned information. |
| Decide / Warrant Gate | Checks whether the proposed action is sufficiently justified and safe. |
| Act and Regulate | Executes the selected action and monitors the process. |
| Record and Justify | Stores information about the request, decisions, actions, and outcomes. |

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.
- Answer: If Eva suddenly stands up while the robot is carrying the mug, the robot should pause its movement and perceive the environment again.

It should update its belief about Eva's position and determine whether the current path is still safe.

If the original path is no longer suitable, the robot should generate an alternative path and check whether it satisfies the safety constraints before continuing.

The robot should only proceed when it has sufficient evidence that the new movement is safe.



## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [6]:

#!/usr/bin/env python3
"""Embodied Cognitive Architecture"""

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional, List
import random
import time
import statistics


@dataclass(frozen=True)
class GoalSpecification:
    """Formalized goal derived from the instruction."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float


@dataclass
class ParameterPacket:
    """Candidate parameters generated by the Proposer."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"
    contact_depth_m: float = 0.015


class Proposer:
    """Generates candidate parameters using uniform sampling."""

    def __init__(self):
        self.min_velocity = 0.2
        self.max_velocity = 0.8
        self.min_tilt_rate = 0.05
        self.max_tilt_rate = 0.2

    def propose(self, grasp_pose=(0.45, 0.12, 0.30),
                velocity=None, tilt_rate=None, contact_depth_m=0.015):
        if velocity is None:
            velocity = random.uniform(
                self.min_velocity, self.max_velocity
            )
        if tilt_rate is None:
            tilt_rate = random.uniform(
                self.min_tilt_rate, self.max_tilt_rate
            )

        return ParameterPacket(
            grasp_pose=grasp_pose,
            max_velocity_mps=round(velocity, 2),
            tilt_rate_radps=round(tilt_rate, 2),
            source_generator="uniform_sampler",
            contact_depth_m=contact_depth_m
        )


@dataclass
class Observation:
    """Raw sensor measurements."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str


@dataclass
class BeliefState:
    """The robot's current internal belief state."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True

    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"
    preferred_cup: str = "blue_ceramic_mug"

    # Battery level as a percentage of total capacity
    battery_level_percent: float = 100.0


class InterpretationSystem:
    """Maps a natural-language instruction to a formal goal."""

    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str,
                  belief: BeliefState) -> GoalSpecification:

        print(
            f"\n[INTERPRETATION] Analyzing instruction: "
            f"'{raw_utterance}' in context of Eva "
            f"(Position: {belief.eva_position}, "
            f"Room Busyness: {belief.room_busyness})"
        )

        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(
            cup_type, {"max_force": 5.0, "max_wave": 2.0}
        )

        busyness_factor = self.kb["room_busyness_limits"].get(
            belief.room_busyness, 0.5
        )
        max_nav_vel = 1.2 * busyness_factor

        is_urgent = (
            "schnell" in raw_utterance.lower()
            or "fast" in raw_utterance.lower()
        )

        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0

        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )

        print(
            f" -> Derived Goal: Object={goal.target_object}, "
            f"MaxForce={goal.max_allowable_force_n}N, "
            f"MaxNavVel={goal.max_nav_velocity_mps}m/s, "
            f"TimeFrame={goal.time_frame_seconds}s"
        )

        return goal


class SensingSystem:
    """Simulates noisy sensor measurements."""

    def sense(self, true_ambient_temp: float,
              true_friction: float, eva_pos: str,
              busyness: str) -> Observation:

        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)

        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Updates the internal belief using sensor observations."""

    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = (
            observation.measured_temperature_c
        )
        self.belief_state.surface_friction_coefficient = (
            observation.measured_friction
        )
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """Predicts physical properties of a proposed action."""

    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket,
                           goal: GoalSpecification) -> Dict[str, float]:

        # Task 2: Dual-stage material compliance
        stiffness_coefficient = (
            12.5 *
            (1.0 + self.belief.surface_friction_coefficient * 0.1)
        )

        if params.contact_depth_m > 0.02:
            stiffness_coefficient *= 1.8

        predicted_force = (
            stiffness_coefficient *
            (params.max_velocity_mps ** 1.5)
        )

        # Task 3: Temperature-dependent fluid viscosity
        temperature_increase = max(
            0.0, self.belief.ambient_temperature_c - 20.0
        )

        effective_viscosity = self.belief.fluid_viscosity_pas * (
            1.0 - temperature_increase * 0.01
        )

        effective_viscosity = max(0.01, effective_viscosity)

        radius = 0.05
        wave_height_m = (
            params.tilt_rate_radps * radius
        ) / effective_viscosity

        predicted_wave = wave_height_m * 1000.0

        # Small prediction errors represent the sim-to-real gap
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        # Task 4: Estimated battery drain
        estimated_battery_drain = params.max_velocity_mps * 30.0

        return {
            "predicted_force_n": round(
                max(0.0, predicted_force + epistemic_force_error), 2
            ),
            "predicted_wave_mm": round(
                max(0.0, predicted_wave + epistemic_wave_error), 2
            ),
            "predicted_nav_velocity": min(
                params.max_velocity_mps, goal.max_nav_velocity_mps
            ),
            "effective_viscosity_pas": round(effective_viscosity, 3),
            "estimated_battery_drain_percent": round(
                estimated_battery_drain, 2
            )
        }


class WarrantGate:
    """Checks whether a proposed action satisfies the constraints."""

    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket,
                 goal: GoalSpecification,
                 belief: BeliefState) -> Dict[str, Any]:

        predictions = self.sdt.predict_properties(params, goal)

        battery_drain = predictions[
            "estimated_battery_drain_percent"
        ]

        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,

            "feasible": (
                params.max_velocity_mps > 0.0
                and params.tilt_rate_radps > 0.0
            ),

            "safe": (
                predictions["predicted_force_n"]
                <= goal.max_allowable_force_n
                and predictions["predicted_wave_mm"]
                <= goal.max_slosh_wave_mm
                and predictions["predicted_nav_velocity"]
                <= goal.max_nav_velocity_mps
            ),

            "monitorable": belief.camera_active,

            "repairable": belief.fallback_path_exists,

            "recordable": belief.audit_logger_enabled,

            # Task 4: Energy must be below 15% and available
            "energetically_viable": (
                battery_drain < 15.0
                and belief.battery_level_percent >= battery_drain
            )
        }

        all_passed = all(checks.values())

        failed_criterion = next(
            (name for name, passed in checks.items() if not passed),
            None
        )

        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Executes approved actions and records the audit trail."""

    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket,
                       goal: GoalSpecification,
                       warrant_result: Dict[str, Any]) -> Dict[str, Any]:

        print(
            f"\n[ACTUATOR] Executing motion in physical world: "
            f"v_max={params.max_velocity_mps} m/s"
        )

        time.sleep(0.1)

        predictions = warrant_result["predictions"]

        # Actual results may differ from the predictions
        actual_force = (
            predictions["predicted_force_n"]
            + random.uniform(-0.3, 0.6)
        )

        actual_wave = (
            predictions["predicted_wave_mm"]
            + random.uniform(-0.2, 0.5)
        )

        spill_occurred = (
            actual_force > goal.max_allowable_force_n
            or actual_wave > goal.max_slosh_wave_mm
        )

        status = "FAILURE" if spill_occurred else "SUCCESS"

        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": (
                goal.target_volume_ml
                if not spill_occurred
                else goal.target_volume_ml - 3.0
            ),
            "spill_occurred": spill_occurred,
            "status": status
        }

        return execution_outcome

    def record_trace(self, asked_utterance: str,
                     goal: GoalSpecification,
                     belief: BeliefState,
                     proposal: ParameterPacket,
                     warrant_result: Dict[str, Any],
                     observation: Observation,
                     committed: bool,
                     execution_outcome: Optional[Dict[str, Any]] = None
                     ) -> None:

        record = {
            "timestamp": time.time(),

            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },

            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient,
                "battery_level_percent": belief.battery_level_percent
            },

            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },

            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps,
                    "contact_depth_m": proposal.contact_depth_m
                },
                "warrant_verdict": warrant_result
            },

            "execution_outcome": execution_outcome
        }

        self.audit_trail.append(record)


# Task 5: Post-hoc audit analysis
def audit_analyzer(audit_trail):

    rejected = 0
    approved = 0
    failed_criteria = {}
    force_errors = []

    for entry in audit_trail:

        committed = entry["committed"]
        warrant = committed["warrant_verdict"]

        outcome = entry["execution_outcome"]

        if committed["action_approved"]:
            if outcome is not None:
                approved += 1
        else:
            rejected += 1

            reason = warrant["failed_criterion"]

            if reason is not None:
                failed_criteria[reason] = (
                    failed_criteria.get(reason, 0) + 1
                )

        if outcome is not None:
            predicted_force = warrant["predictions"]["predicted_force_n"]
            actual_force = outcome["actual_force_n"]

            force_errors.append(actual_force - predicted_force)

    print("\n===== POST-HOC AUDIT ANALYSIS =====")
    print(f"Rejected proposals: {rejected}")
    print(f"Approved executions: {approved}")

    if failed_criteria:
        most_common = max(failed_criteria, key=failed_criteria.get)
        print(f"Most frequent rejection criterion: {most_common}")
    else:
        print("No rejected proposals.")

    if force_errors:
        force_variance = statistics.pvariance(force_errors)
        average_error = sum(
            abs(error) for error in force_errors
        ) / len(force_errors)

        print(
            f"Average absolute force prediction error: "
            f"{average_error:.3f} N"
        )
        print(
            f"Variance of force prediction errors: "
            f"{force_variance:.4f} N^2"
        )
    else:
        print("No physical force outcomes available for comparison.")


# --------------------------------------------------
# Execution Demonstration
# --------------------------------------------------

if __name__ == "__main__":

    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {
                "max_force": 4.5,
                "max_wave": 1.5
            },
            "plastic_cup": {
                "max_force": 7.0,
                "max_wave": 3.0
            }
        },

        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(
        room_busyness="medium",
        preferred_cup="blue_ceramic_mug"
    )

    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    proposer = Proposer()

    print("=== STEP 1: Sensing & Perception Update ===")

    raw_observation = sensing.sense(
        true_ambient_temp=21.5,
        true_friction=0.45,
        eva_pos="living_room",
        busyness="medium"
    )

    perception.update(raw_observation)

    print(f"Raw Sensor Observation : {raw_observation}")
    print(
        f"Updated Internal Belief : "
        f"Temp={belief_state.ambient_temperature_c}°C, "
        f"Friction={belief_state.surface_friction_coefficient}"
    )

    print("\n=== STEP 2: Instruction Interpretation ===")

    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)

    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")

    # Show one sample from the uniform proposer
    sample_proposal = proposer.propose()

    print(f"Proposer sample: {sample_proposal}")

    # First attempt: deliberately aggressive proposal
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8,
        tilt_rate_radps=0.2,
        contact_depth_m=0.025
    )

    print(
        f"First proposal: v_max = "
        f"{proposal_v1.max_velocity_mps} m/s"
    )

    print(
        "\n=== STEP 4: SDT & Warrant Gate Validation "
        "(Attempt 1) ==="
    )

    warrant_v1 = warrant_gate.validate(
        proposal_v1, goal, belief_state
    )

    print(f"SDT Predictions: {warrant_v1['predictions']}")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")

    verdict_v1 = (
        "APPROVED" if warrant_v1["passed"] else "REJECTED"
    )

    print(f"Gate Verdict    : {verdict_v1}")

    executive.record_trace(
        raw_command, goal, belief_state,
        proposal_v1, warrant_v1, raw_observation,
        committed=warrant_v1["passed"]
    )

    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")

    proposal_v2 = proposer.propose(
        velocity=0.45,
        tilt_rate=0.03,
        contact_depth_m=0.015
    )

    warrant_v2 = warrant_gate.validate(
        proposal_v2, goal, belief_state
    )

    print(f"SDT Predictions: {warrant_v2['predictions']}")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")

    verdict_v2 = (
        "APPROVED" if warrant_v2["passed"] else "REJECTED"
    )

    print(f"Gate Verdict    : {verdict_v2}")

    print("\n=== STEP 6: Physical Execution ===")

    if warrant_v2["passed"]:

        final_outcome = executive.execute_action(
            proposal_v2, goal, warrant_v2
        )

        executive.record_trace(
            raw_command, goal, belief_state,
            proposal_v2, warrant_v2, raw_observation,
            committed=True,
            execution_outcome=final_outcome
        )

        print(f"Execution Outcome: {final_outcome}")

    else:
        print("Action was not executed because the gate rejected it.")

    # Task 2.2: Test aggressive approach velocity
    print("\n=== TASK 2.2: Aggressive Contact Test ===")

    aggressive_proposal = proposer.propose(
        velocity=0.35,
        tilt_rate=0.08,
        contact_depth_m=0.025
    )

    aggressive_result = warrant_gate.validate(
        aggressive_proposal, goal, belief_state
    )

    print(
        "Aggressive predicted force:",
        aggressive_result["predictions"]["predicted_force_n"], "N"
    )
    print(
        "Aggressive warrant checks:",
        aggressive_result["warrant_checks"]
    )
    print(
        "Aggressive proposal:",
        "APPROVED" if aggressive_result["passed"] else "REJECTED"
    )

    # Task 3.2: Temperature test
    print("\n=== TASK 3.2: Temperature and Sloshing Test ===")

    old_temperature = belief_state.ambient_temperature_c
    belief_state.ambient_temperature_c = 28.0

    hot_proposal = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.30,
        tilt_rate_radps=0.15,
        contact_depth_m=0.015
    )

    hot_result = warrant_gate.validate(
        hot_proposal, goal, belief_state
    )

    print(
        "Effective viscosity:",
        hot_result["predictions"]["effective_viscosity_pas"], "Pa.s"
    )
    print(
        "Predicted wave height:",
        hot_result["predictions"]["predicted_wave_mm"], "mm"
    )
    print("Warrant checks:", hot_result["warrant_checks"])
    print(
        "Temperature test:",
        "APPROVED" if hot_result["passed"] else "REJECTED"
    )

    belief_state.ambient_temperature_c = old_temperature

    # Task 4.2: Low battery test
    print("\n=== TASK 4.2: Low Battery Test ===")

    old_battery = belief_state.battery_level_percent
    belief_state.battery_level_percent = 10.0

    low_battery_result = warrant_gate.validate(
        proposal_v2, goal, belief_state
    )

    print(
        "Estimated battery drain:",
        low_battery_result["predictions"][
            "estimated_battery_drain_percent"
        ], "%"
    )
    print("Warrant checks:", low_battery_result["warrant_checks"])

    if not low_battery_result["warrant_checks"]["energetically_viable"]:
        next_action = "request_recharge"
        print(
            "Battery is too low. Workflow redirected to:",
            next_action
        )
    elif low_battery_result["passed"]:
        print("Battery level is sufficient. The robot may proceed.")

    belief_state.battery_level_percent = old_battery

    # Task 5: Analyse the recorded execution history
    print(
        f"\n--- COMPLETE EPISODIC AUDIT TRAIL "
        f"({len(executive.audit_trail)} Entries) ---"
    )

    for idx, trace in enumerate(executive.audit_trail, 1):

        status = trace["committed"]["action_approved"]
        outcome = trace.get("execution_outcome")

        outcome_str = (
            f" | Outcome: {outcome['status']} "
            f"(Spill: {outcome['spill_occurred']})"
            if outcome else ""
        )

        print(
            f"Entry #{idx} | Approved: {status}"
            f"{outcome_str} | Command: "
            f"'{trace['asked']['raw_utterance']}'"
        )

    audit_analyzer(executive.audit_trail)


=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793488255.4791882, measured_temperature_c=21.16, measured_friction=0.48, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.16°C, Friction=0.48

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer sample: ParameterPacket(grasp_pose=(0.45, 0.12, 0.3), max_velocity_mps=0.41, tilt_rate_radps=0.15, source_generator='uniform_sampler', contact_depth_m=0.015)
First proposal: v_max = 0.8 m/s

=== STEP 4: SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: {'predicted_force_n': 16.69, 'predicted_wave_mm': 8.51, 'predicted_nav_velocity': 0.72, 'effective_viscosity_pas': 1.186, 'estim

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition

**Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.

**Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

### Task 1.1 – Architectural Mapping and Code Decomposition

The Python script implements an embodied cognitive architecture in which different classes perform specific functions. Each class corresponds to one or more functional blocks in the lecture's integration diagram.

| Functional Block | Python Class / Method | Responsibility |
|---|---|---|
| Interpret and Ground | `InterpretationSystem.interpret()` | Converts the natural-language instruction into a formal `GoalSpecification` using the knowledge base and current belief state. |
| Know and Predict | `SemanticDigitalTwin.predict_properties()` | Predicts physical consequences such as contact force, fluid wave height and navigation velocity. |
| Perceive | `SensingSystem.sense()` | Collects noisy measurements from the simulated sensors. |
| Perceive | `PerceptionSystem.update()` | Processes observations and updates the internal `BeliefState`. |
| Propose | `Proposer.propose()` / `ParameterPacket` | Generates candidate execution parameters such as grasp pose, velocity and tilt rate using uniform sampling. Controlled parameter values can also be supplied for the assignment test cases. |
| Decide – Warrant Gate | `WarrantGate.validate()` | Checks whether the proposed action satisfies the grounded, feasible, safe, monitorable, repairable and recordable criteria. |
| Act and Regulate | `ExecutionEngine.execute_action()` | Executes an approved action and simulates its physical outcome. |
| Record and Justify | `ExecutionEngine.record_trace()` | Records the original request, internal belief, observations, commitment decision and execution outcome. |

#### Data Flow During a Cognitive Decision Cycle

The decision cycle begins when Eva gives the instruction, "Bring me my coffee."

1. **Perception:** `SensingSystem.sense()` collects environmental measurements, such as temperature, friction, Eva's position and room busyness.

2. **Belief Update:** `PerceptionSystem.update()` uses the observations to update the robot's internal `BeliefState`.

3. **Interpretation:** `InterpretationSystem.interpret()` processes Eva's instruction and the current belief state to produce a formal `GoalSpecification`.

4. **Proposal:** `Proposer.propose()` generates candidate movement parameters using uniform sampling and stores them in a `ParameterPacket`. For the assignment test cases, specific values can also be supplied to test particular conditions.

5. **Prediction:** `SemanticDigitalTwin.predict_properties()` estimates the physical consequences of the proposed movement.

6. **Decision:** `WarrantGate.validate()` compares the predictions with the constraints in the goal specification. It either approves or rejects the proposal.

7. **Execution:** If the proposal passes validation, `ExecutionEngine.execute_action()` carries out the action and produces an execution outcome.

8. **Recording:** `ExecutionEngine.record_trace()` stores the request, belief, observations, decision and outcome in the audit trail.

This architecture separates proposing an action from approving and executing it. A candidate action must pass the Warrant Gate before the robot can execute it.

### Question 1.2

I implemented the Proposer using a uniform probability distribution.

The velocity is randomly selected between 0.2 and 0.8 m/s, while the tilt rate is selected between 0.05 and 0.2 rad/s.

The generated values are stored in a `ParameterPacket` together with the grasp pose.

I chose uniform sampling because it is simple to implement and allows the robot to explore different possible movement parameters. Every value within the selected range has an equal chance of being generated.

For the assignment test cases, the Proposer can also receive specific velocity and tilt values. This makes it possible to test controlled situations while retaining uniform sampling for normal proposals.

---

## Task 2: Non-Linear Contact Mechanics and Material Compliance

In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation.

**Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds 0.02 m, the stiffness coefficient should increase exponentially by a factor of 1.8 due to bottoming out against a rigid internal fixture.

**Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity (0.35 m/s). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

## Task 2 Answer: Non-Linear Contact Mechanics and Material Compliance

### Question 2.1

The original contact model used a single stiffness coefficient. I modified it to account for two stages of material compliance.

Initially, the material behaves as a flexible surface. When the contact depth exceeds 0.02 m, the pouch reaches a rigid internal fixture. At this point, the stiffness coefficient is multiplied by 1.8.

The modified model is:

- If contact depth is less than or equal to 0.02 m, the original stiffness is used.
- If contact depth exceeds 0.02 m, the stiffness is increased by a factor of 1.8.

This represents the change in material behaviour when the flexible pouch reaches its rigid internal support.

### Question 2.2

For the test case, I used the Proposer to generate an aggressive parameter packet with an approach velocity of 0.35 m/s and a contact depth of 0.025 m.

Since the contact depth is greater than 0.02 m, the stiffness coefficient is multiplied by 1.8.

Using the original force model:

\[
F = k \times v^{1.5}
\]

The modified stiffness is:

\[
k = 12.5 \times (1 + 0.45 \times 0.1) \times 1.8
\]

\[
k = 23.5125
\]

Therefore:

\[
F = 23.5125 \times (0.35)^{1.5}
\]

\[
F \approx 4.87 \text{ N}
\]

The maximum allowable force for the blue ceramic mug is 4.5 N.

Since the predicted force exceeds this limit, the Warrant Gate rejects the aggressive proposal before execution.

The exact predicted value can vary slightly because the code includes sensor noise and a small prediction error. However, the force remains above the 4.5 N limit, so the Warrant Gate rejects the proposal before execution.

---

## Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling

Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.

**Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient (1% drop per °C above 20°C).

**Question 3.2:** Simulate an environment where the ambient temperature rises to 28°C. How does this change the predicted wave height for a tilt rate of 0.15 rad/s? Does the Warrant Gate trigger a safety violation?

## Task 3 Answer: Advanced Fluid Sloshing and Viscosity Temperature Coupling

### Question 3.1

The fluid viscosity model was modified to account for changes in ambient temperature.

A linear thermal degradation coefficient of 1% per degree Celsius above 20°C is used.

The effective viscosity is calculated as:

\[
\eta = \eta_0 \times (1 - 0.01 \times (T - 20))
\]

where:

- \(\eta\) is the effective viscosity.
- \(\eta_0\) is the original fluid viscosity.
- \(T\) is the ambient temperature in degrees Celsius.

The temperature increase reduces the viscosity of the fluid. Lower viscosity means that the liquid can move more easily, which can increase the predicted wave height.

### Question 3.2

For this test, the ambient temperature is increased to 28°C.

The original viscosity is 1.2 Pa·s.

Effective viscosity:

\[
\eta = 1.2 \times (1 - 0.01 \times (28 - 20))
\]

\[
\eta = 1.2 \times 0.92
\]

\[
\eta = 1.104 \text{ Pa·s}
\]

The tilt rate is 0.15 rad/s and the radius is 0.05 m.

The predicted wave height is calculated using:

\[
\text{Wave height} = \left(\frac{\text{Tilt rate} \times \text{Radius}}{\text{Effective viscosity}}\right) \times 1000
\]

\[
\text{Wave height} = \left(\frac{0.15 \times 0.05}{1.104}\right) \times 1000
\]

\[
\text{Wave height} \approx 6.79 \text{ mm}
\]

The maximum allowable wave height for the blue ceramic mug is 1.5 mm.

Since 6.79 mm is greater than 1.5 mm, the Warrant Gate triggers a safety violation and rejects the proposal.

This shows how an increase in temperature can reduce fluid viscosity and increase the risk of spilling.

---

## Task 4: Expanding the Warrant Gate Checklist

The current Warrant Gate evaluates six core criteria (**grounded, feasible, safe, monitorable, repairable, recordable**).

**Question 4.1:** Add a seventh criterion called `energetically_viable` to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below 15% of total capacity.

**Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

## Task 4 Answer: Expanding the Warrant Gate Checklist

### Question 4.1

A seventh criterion called `energetically_viable` was added to the Warrant Gate.

The estimated battery drain is calculated using the proposed velocity:

\[
\text{Battery drain (\%)} = \text{Maximum velocity} \times 30
\]

The proposal passes this criterion only when:

1. The estimated battery drain is below 15% of the total battery capacity.
2. The available battery level is sufficient to cover the estimated drain.

This check prevents the robot from starting a task when it does not have enough energy to complete it.

### Question 4.2

A battery level of 10% was introduced into the `BeliefState` to simulate a low-battery situation.

For a proposed velocity of 0.45 m/s:

Estimated battery drain:

\[
0.45 \times 30 = 13.5\%
\]

Although the estimated drain is below the 15% limit, the robot only has 10% battery remaining.

Therefore, the `energetically_viable` criterion fails because the available battery is insufficient.

Instead of proceeding with coffee delivery, the system requests a recharge action.

This allows the robot to consider its available energy before committing to a task.

---

## Task 5: Post-Hoc Epistemic Audit Analysis

A core pillar of trustworthy robotics is the ability to audit **why** a decision was made or rejected.

**Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:

1. The total number of rejected proposals versus approved executions.
2. The most frequent warrant criterion responsible for rejection.
3. The average variance between predicted forces and observed physical outcomes.

**Question 5.2:** Discuss in your lab report: **Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?**

## Task 5 Answer: Post-Hoc Epistemic Audit Analysis

### Question 5.1

The `audit_analyzer(audit_trail)` function processes the recorded execution history.

It calculates:

1. The total number of rejected proposals.
2. The total number of approved executions.
3. The most frequent criterion responsible for rejection.
4. The average absolute difference and variance between predicted and actual contact forces.

The function uses the information stored in the `committed` and `execution_outcome` sections of each audit entry. An approved execution is counted only when the proposal passed the Warrant Gate and an execution outcome was recorded.

The force prediction error is calculated by subtracting the predicted force from the actual measured force.

The variance of these errors helps show how much the prediction errors differ across recorded executions. In the current demonstration, only one physical execution is recorded, so the population variance is 0.0000 N². More execution samples are needed for a meaningful comparison of variance.

### Question 5.2

Maintaining separate records for `asked`, `belief`, `committed` and `observed` is important because each represents a different part of the robot's decision process.

- **Asked:** Stores the original instruction given by the user and the interpreted goal.
- **Belief:** Stores what the robot currently believes about the environment.
- **Committed:** Stores the proposed parameters and the result of the Warrant Gate.
- **Observed:** Stores the actual sensor measurements collected from the environment.

These records make it possible to compare what the user requested, what the robot believed, what it decided to do and what it actually observed.

Pure data-driven imitation learning mainly learns patterns from previous demonstrations. It does not necessarily explain why an action was selected or whether the action was suitable in a new situation.

Keeping these records separate makes the robot's decisions easier to inspect and understand. It also helps identify differences between predictions and actual outcomes, which can be used to improve the model and future decisions.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

## Part 3: Robot Anatomy (0.5 Pt)

A robot consists of different mechanical and electronic components that work together to perform tasks. The PR2 is a mobile manipulator robot with a mobile base, torso, head, two arms and grippers.

### Key Functional Parts of a Robot

| Component | Explanation | Function in the Robot |
|---|---|---|
| Links | Rigid parts of a robot connected by joints. | Form the physical structure of the robot's arms and body. |
| Joints | Connections between two links that allow movement. | Allow the robot's arms, wrists and other parts to move. |
| Actuators | Components that produce movement, usually using motors. | Move the joints and control the robot's movements. |
| Sensors | Components that collect information from the robot or its surroundings. | Help the robot understand its position, objects and environment. |
| Effectors | Parts that directly interact with the environment. | Perform actions such as grasping, holding and moving objects. |
| Base | The bottom part that supports the robot and enables mobility. | Allows the robot to move around the environment. |
| Torso | The main body section of the robot. | Supports the head and arms. |
| Head | The upper part of the robot that usually contains cameras and other sensors. | Helps the robot observe its surroundings. |
| Wheels | Circular components used for movement. | Allow the PR2 to move forward, backward and turn. |
| Legs | Mechanical structures used for walking. | Support walking robots and help them move across different surfaces. |
| Grippers | Mechanical devices attached to the arms. | Allow the robot to pick up, hold and release objects. |

### Types of Sensors

| Sensor Information | Explanation | Example |
|---|---|---|
| Proprioception | Information about the robot's own body and movements. | Joint position, joint velocity and wheel rotation. |
| Exteroception | Information collected from the external environment. | Cameras and laser scanners detecting objects and obstacles. |
| Interoception | Information about the robot's internal condition. | Motor temperature, battery level and internal system status. |

### How These Components Support the Architecture

For Eva's request, "Bring me my coffee", the robot uses these components together:

1. **Perception:** Cameras and other external sensors detect the coffee mug and its position.
2. **Navigation:** The mobile base and wheels allow the robot to move towards the mug while avoiding obstacles.
3. **Manipulation:** The actuators move the arm joints, and the grippers hold the mug.
4. **Feedback:** Proprioceptive sensors provide information about the robot's movements, while other sensors monitor the environment.
5. **Execution:** The robot uses its body and effectors to pick up the mug and deliver it to Eva.

### Conclusion

All these components work together to allow a robot to interact with the physical world. Sensors provide information, actuators produce movement, and effectors perform actions. The robot uses this information through its control architecture to plan and execute tasks safely.

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 